# Project: U-Net from scratch 🎨
Build the encoder–decoder network behind most of medical and satellite image segmentation, U-Net (Ronneberger et al. 2015), and train it to colour in **every pixel** of synthetic images containing circles, squares and triangles on a textured, noisy background. Then remove the skip connections and measure exactly what they buy you: sharper boundaries.

Topic: [[Image Segmentation]], [[CNN Architectures]] · exercises: [Image Segmentation - Exercises](Image%20Segmentation%20-%20Exercises.ipynb) · guide note: [[Project - U-Net from Scratch]] · paper: [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)

Runtime ≈ 2 minutes on a CPU.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import binary_dilation, binary_erosion
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
S, N_CLASSES = 64, 4                                   # classes: 0 background, 1 circle, 2 square, 3 triangle
def make_dataset(n, rng):
    X = np.zeros((n, 1, S, S), np.float32); Y = np.zeros((n, S, S), np.int64); yy, xx = np.mgrid[0:S, 0:S]
    for k in range(n):
        img = rng.normal(0.3, 0.15, (S, S)) + 0.1 * np.sin(xx / rng.uniform(3, 8)); lab = np.zeros((S, S), int)
        for _ in range(rng.integers(2, 5)):
            c = rng.integers(1, 4); cy, cx = rng.integers(8, S - 8, 2); r = rng.integers(5, 12)
            if c == 1: m = (yy - cy) ** 2 + (xx - cx) ** 2 <= r * r
            elif c == 2: m = (abs(yy - cy) <= r * 0.8) & (abs(xx - cx) <= r * 0.8)
            else: m = (yy - cy >= -r) & (yy - cy <= r) & (abs(xx - cx) <= (yy - cy + r) / 2)
            lab[m] = c; img[m] = rng.normal(0.7, 0.15, m.sum())          # shapes are brighter, but just as noisy
        X[k, 0] = img; Y[k] = lab
    return torch.tensor(X), torch.tensor(Y)
rng = np.random.default_rng(0); X_tr, Y_tr = make_dataset(800, rng); X_te, Y_te = make_dataset(200, rng)
fig, ax = plt.subplots(2, 5, figsize=(13, 5.2))
for j in range(5):
    ax[0, j].imshow(X_tr[j, 0], cmap="gray"); ax[1, j].imshow(Y_tr[j], cmap="tab10", vmin=0, vmax=9); ax[0, j].axis("off"); ax[1, j].axis("off")
ax[0, 0].set_title("input"); ax[1, 0].set_title("target mask"); plt.show()
print(f"background pixels: {(Y_tr == 0).float().mean():.1%}  ← an 'all background' model already gets this pixel accuracy")

## 1. Soft Dice loss (multi-class)
Cross-entropy is dominated by the many background pixels. **Dice loss** measures overlap per class: softmax the logits, one-hot the target, and for each class $c$ compute $D_c = \frac{2\sum p_c g_c + \epsilon}{\sum p_c + \sum g_c + \epsilon}$ (sums over batch and pixels). Return $1 - \text{mean}_{c\ge1} D_c$ (foreground classes only). We'll train with cross-entropy + Dice.

In [ ]:
def dice_loss(logits, target, n_classes=N_CLASSES, eps=1.0):
    # TODO 1: softmax probs, one-hot target (B, C, H, W), per-class soft Dice, 1 − mean over classes 1..C-1
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_dice():
    tgt = torch.randint(0, 4, (2, 8, 8)); perfect = F.one_hot(tgt, 4).permute(0, 3, 1, 2).float() * 50
    wrong = F.one_hot((tgt + 1) % 4, 4).permute(0, 3, 1, 2).float() * 50
    return dice_loss(perfect, tgt).item() < 0.01 and dice_loss(wrong, tgt).item() > 0.95 and dice_loss(torch.randn(2, 4, 8, 8, requires_grad=True), tgt).requires_grad
check("Dice loss", _t_dice, "Confident correct logits → ≈ 0; confidently wrong → ≈ 1; the loss must be differentiable.")

## 2. The U-Net
Two down-sampling stages (conv block → max-pool), a bottleneck, and two up-sampling stages (2×2 transposed conv → conv block). With `skips=True` each decoder stage **concatenates** the up-sampled features with the encoder features of the same resolution (`torch.cat([up, enc], dim=1)`); with `skips=False` it uses the up-sampled features alone. The blocks and channel sizes are set up for you; write `forward`.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, skips=True, ch=(16, 32, 64), n_classes=N_CLASSES):
        super().__init__(); self.skips = skips; k = 2 if skips else 1
        self.enc1, self.enc2, self.bottleneck = conv_block(1, ch[0]), conv_block(ch[0], ch[1]), conv_block(ch[1], ch[2])
        self.up2, self.dec2 = nn.ConvTranspose2d(ch[2], ch[1], 2, stride=2), conv_block(ch[1] * k, ch[1])
        self.up1, self.dec1 = nn.ConvTranspose2d(ch[1], ch[0], 2, stride=2), conv_block(ch[0] * k, ch[0])
        self.head = nn.Conv2d(ch[0], n_classes, 1)
    def forward(self, x):
        # TODO 2: enc1 → pool → enc2 → pool → bottleneck → up2 (+ skip e2) → dec2 → up1 (+ skip e1) → dec1 → head
        raise NotImplementedError  # TODO 2

In [ ]:
def _t_unet():
    x = torch.randn(2, 1, 64, 64); a, n = UNet(True), UNet(False)
    out = a(x); x2 = x.clone(); x2[:, :, 0, 0] += 10                  # a change in one corner pixel …
    return out.shape == (2, 4, 64, 64) and n(x).shape == (2, 4, 64, 64) and not torch.allclose(a.eval()(x2)[:, :, 0, 0], a.eval()(x)[:, :, 0, 0])
check("U-Net forward", _t_unet, "Output must be (B, 4, 64, 64) for both variants, and the prediction at a pixel must react to that pixel.")

## 3. Mean IoU over the foreground classes
For each class 1…C−1 present in prediction or target: IoU = |pred ∩ target| / |pred ∪ target|. Return the mean.

In [ ]:
def mean_iou(pred, target, n_classes=N_CLASSES):
    # TODO 3: per-class IoU for classes 1..C-1 (skip classes absent from both), mean
    raise NotImplementedError  # TODO 3

In [ ]:
check("mIoU", lambda: abs(mean_iou(torch.tensor([[0, 1, 1], [2, 2, 0]]), torch.tensor([[0, 1, 0], [2, 2, 2]])) - np.mean([1 / 2, 2 / 3])) < 1e-9,
      "Class 1: 1/2, class 2: 2/3, class 3 absent → mean 0.583.")
def boundary_accuracy(pred, target, width=2):
    """Accuracy on pixels within `width` px of a shape edge – where skip connections should help."""
    g, p = target.numpy(), pred.numpy(); edge = np.zeros_like(g, bool)
    for k in range(len(g)):
        for c in range(1, N_CLASSES):
            m = g[k] == c; edge[k] |= binary_dilation(m, iterations=width) & ~binary_erosion(m, iterations=width)
    return float((p[edge] == g[edge]).mean())

## 4. Train with and without skip connections
Same architecture, same data, same seed, cross-entropy + Dice, 20 epochs each.

In [ ]:
EPOCHS = 20; results, models = {}, {}
if ready("Dice loss", "U-Net forward", "mIoU"):
    for skips in (True, False):
        torch.manual_seed(0); model = UNet(skips); opt = torch.optim.Adam(model.parameters(), 2e-3); t = time.time()
        for epoch in range(EPOCHS):
            model.train()
            for idx in torch.randperm(len(X_tr)).split(32):
                logits = model(X_tr[idx]); loss = F.cross_entropy(logits, Y_tr[idx]) + dice_loss(logits, Y_tr[idx])
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): pred = model(X_te).argmax(1)
        name = "U-Net (skips)" if skips else "no skips"; models[name] = model
        results[name] = dict(mIoU=mean_iou(pred, Y_te), pixel_acc=(pred == Y_te).float().mean().item(), boundary_acc=boundary_accuracy(pred, Y_te))
        print(f"{name:<14} mIoU {results[name]['mIoU']:.3f}  pixel acc {results[name]['pixel_acc']:.3f}  boundary acc {results[name]['boundary_acc']:.3f}  ({time.time() - t:.0f}s)")

In [ ]:
if ready("Dice loss", "U-Net forward", "mIoU"):
    with torch.no_grad(): pa, pb = models["U-Net (skips)"](X_te[:4]).argmax(1), models["no skips"](X_te[:4]).argmax(1)
    fig, ax = plt.subplots(4, 4, figsize=(10, 10))
    for i in range(4):
        for j, (img, t) in enumerate([(X_te[i, 0], "input"), (Y_te[i], "truth"), (pa[i], "U-Net"), (pb[i], "no skips")]):
            ax[i, j].imshow(img, cmap="gray" if j == 0 else "tab10", vmin=0 if j else None, vmax=9 if j else None); ax[i, j].axis("off")
            if i == 0: ax[i, j].set_title(t)
    plt.show()
    u, n = results["U-Net (skips)"], results["no skips"]
    print(f"boundary errors: U-Net {1 - u['boundary_acc']:.1%} vs no skips {1 - n['boundary_acc']:.1%}")
    check("skip connections sharpen boundaries", lambda: u["mIoU"] > 0.85 and u["mIoU"] > n["mIoU"] and u["boundary_acc"] > n["boundary_acc"],
          "The U-Net should reach mIoU > 0.85 and beat the no-skip version on mIoU and boundary accuracy.")

<details><summary>▶ Solution</summary>

```python
def dice_loss(logits, target, n_classes=N_CLASSES, eps=1.0):
    probs = F.softmax(logits, dim=1)
    onehot = F.one_hot(target, n_classes).permute(0, 3, 1, 2).float()
    inter = (probs * onehot).sum((0, 2, 3)); total = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    dice = (2 * inter + eps) / (total + eps)
    return 1 - dice[1:].mean()
```

```python
    def forward(self, x):   # UNet
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        b = self.bottleneck(F.max_pool2d(e2, 2))
        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1) if self.skips else u2)
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1) if self.skips else u1)
        return self.head(d1)
```

```python
def mean_iou(pred, target, n_classes=N_CLASSES):
    ious = []
    for c in range(1, n_classes):
        inter = ((pred == c) & (target == c)).sum().item(); union = ((pred == c) | (target == c)).sum().item()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious))
```
</details>

## Stretch goals
- Train with cross-entropy only vs. CE + Dice vs. Dice only. Which helps the small shapes most?
- Make shapes touch and count them with connected components (Exercise 9 of the exercises): when does semantic segmentation fail at counting?
- Replace the transposed convolutions with bilinear upsampling + conv (no checkerboard artefacts).
- Real data: the Oxford-IIIT Pet dataset (`torchvision.datasets.OxfordIIITPet`, segmentation masks) with the same U-Net.